# Aula 06 — Observabilidade por *tracing* com MLflow

**Deploy de Agentes com LLMs** · Luiz Fernando Braz

A validação responde *"a resposta foi boa?"*. A **observabilidade** responde uma pergunta
anterior e mais básica: **"o que exatamente o agente fez?"**. Sem isso, depurar uma falha em
produção é adivinhação.

A resposta da indústria é o **tracing distribuído**, e o padrão que emergiu para LLMs em
2024–2026 são as **GenAI semantic conventions do OpenTelemetry** — um vocabulário comum
(`gen_ai.*`) para descrever o que acontece dentro de uma chamada de IA generativa. Cada
execução vira uma **árvore de *spans*** (trechos cronometrados de trabalho):

- **`invoke_agent`** (span de topo) — a execução inteira do agente;
- **`chat`** — cada chamada ao LLM;
- **`execute_tool`** — cada uso de ferramenta.

O **MLflow Tracing** implementa essas convenções: captura latência e tokens em cada passo, e o
**mesmo trace** alimenta depois o `mlflow.genai.evaluate()` (ver `exercicio_eval_observabilidade.ipynb`).

> **Modo de execução.** As seções abaixo rodam **offline**: um *tracer* didático **espelha** o
> MLflow Tracing (`start_span`, `span_type`, atributos `gen_ai.*`). No final há uma **seção
> opcional** que grava os mesmos spans no MLflow de verdade — aí sim você abre a UI. Em produção,
> basta uma linha de *autolog* e o MLflow instrumenta o agente sozinho.

In [ ]:
import contextlib
from dataclasses import dataclass, field

# Em produção, o MLflow Tracing faz tudo isto automaticamente:
#   import mlflow
#   mlflow.anthropic.autolog()    # (ou mlflow.langchain.autolog(), mlflow.openai.autolog())
# A partir daí, cada chamada ao Claude/ferramenta já vira um span — sem mais código.

# Espelha mlflow.entities.SpanType — os tipos de span padronizados.
class SpanType:
    AGENT = "AGENT"; LLM = "LLM"; CHAT_MODEL = "CHAT_MODEL"; TOOL = "TOOL"
    RETRIEVER = "RETRIEVER"; CHAIN = "CHAIN"

# Espelha um span do MLflow Tracing: nome, tipo, I/O, atributos e duração.
@dataclass
class Span:
    name: str
    span_type: str = "UNKNOWN"
    inputs: object = None
    outputs: object = None
    attributes: dict = field(default_factory=dict)
    latency_ms: float = 0.0
    children: list = field(default_factory=list)
    def set_inputs(self, v): self.inputs = v
    def set_outputs(self, v): self.outputs = v
    def set_attributes(self, d): self.attributes.update(d)

# Espelha mlflow.start_span(...) + o registro de traces (uma pilha de spans aninhados).
class Tracer:
    def __init__(self): self.traces, self._stack = [], []
    @contextlib.contextmanager
    def start_span(self, name, span_type="UNKNOWN"):
        sp = Span(name, span_type)
        (self._stack[-1].children if self._stack else self.traces).append(sp)
        self._stack.append(sp)
        try:
            yield sp
        finally:
            self._stack.pop()
    def reset(self): self.traces, self._stack = [], []

tracer = Tracer()
print("Tracer pronto — espelha o MLflow Tracing (start_span, span_type, atributos gen_ai.*).")

## Instrumentando um agente: a árvore de *spans*

Vamos rastrear um agente de atendimento que responde sobre a previsão do tempo da cidade de um
pedido. A execução gera um span de topo **`invoke_agent`** com filhos: um **`chat`** em que o
LLM **decide chamar a ferramenta** (`finish_reason = tool_calls`), um **`execute_tool`** e um
segundo **`chat`** que **redige a resposta** (`finish_reason = stop`).

```python
# PRODUÇÃO — instrumentação manual (o autolog dispensa até isto):
import mlflow

@mlflow.trace(span_type="AGENT")          # span de topo 'invoke_agent'
def responder(pergunta, cidade):
    ...
    with mlflow.start_span(name="execute_tool", span_type="TOOL") as span:
        span.set_inputs({"cidade": cidade})
        span.set_outputs(consultar_clima(cidade))
```

In [ ]:
def consultar_clima(cidade):
    return {"previsao": "sol", "cidade": cidade}

def responder(pergunta, cidade, falhar=False, ruido_ms=0):
    """Agente instrumentado. `falhar=True` simula timeout na ferramenta (span de erro)."""
    with tracer.start_span("invoke_agent", SpanType.AGENT) as root:
        root.set_inputs({"pergunta": pergunta, "cidade": cidade})

        # 1) o LLM decide chamar a ferramenta -> finish_reason = tool_calls
        with tracer.start_span("chat", SpanType.CHAT_MODEL) as s1:
            s1.set_attributes({"gen_ai.request.model": "claude-sonnet-4-6",
                               "gen_ai.usage.input_tokens": 180,
                               "gen_ai.usage.output_tokens": 22,
                               "gen_ai.response.finish_reasons": ["tool_calls"]})
            s1.latency_ms = 520

        # 2) a ferramenta executa
        with tracer.start_span("execute_tool", SpanType.TOOL) as s2:
            s2.set_inputs({"cidade": cidade})
            s2.set_attributes({"gen_ai.tool.name": "consultar_clima"})
            if falhar:
                s2.set_attributes({"error.type": "TimeoutError"})
                s2.latency_ms = 3000
                root.set_outputs({"resposta": "Não consegui obter a previsão agora."})
                root.latency_ms = s1.latency_ms + s2.latency_ms
                return root.outputs["resposta"]
            s2.set_outputs(consultar_clima(cidade))
            s2.latency_ms = 95

        # 3) o LLM redige a resposta final -> finish_reason = stop
        with tracer.start_span("chat", SpanType.CHAT_MODEL) as s3:
            s3.set_attributes({"gen_ai.request.model": "claude-sonnet-4-6",
                               "gen_ai.usage.input_tokens": 240,
                               "gen_ai.usage.output_tokens": 34,
                               "gen_ai.response.finish_reasons": ["stop"]})
            s3.latency_ms = 410 + ruido_ms

        resposta = f"Em {cidade}, a previsão é de sol. Bom passeio!"
        root.set_outputs({"resposta": resposta})
        root.latency_ms = s1.latency_ms + s2.latency_ms + s3.latency_ms
        return resposta

print(responder("Como está o tempo na cidade do meu pedido?", "Recife"))

## Lendo o trace de volta: os atributos `gen_ai.*`

Cada span carrega **atributos `gen_ai.*` bem definidos**, o "raio-x" do raciocínio:

- **`gen_ai.request.model`** — o modelo usado (ex.: `claude-sonnet-4-6`);
- **`gen_ai.usage.input_tokens`** / **`output_tokens`** — a contagem de tokens, **base do custo**;
- **`gen_ai.response.finish_reasons`** — por que o modelo parou (`stop` ou `tool_calls`, sinal
  precioso de **quando o agente decidiu chamar uma ferramenta**);
- **`gen_ai.tool.name`** — a ferramenta executada.

Quando a captura de conteúdo está habilitada, registram-se ainda `gen_ai.input.messages` e
`gen_ai.output.messages`. Vamos percorrer a árvore capturada e imprimi-la.

In [ ]:
def imprimir_arvore(span, nivel=0):
    ind = "    " * nivel
    a = span.attributes
    extra = ""
    if "gen_ai.request.model" in a:
        ti = a.get("gen_ai.usage.input_tokens", 0)
        to = a.get("gen_ai.usage.output_tokens", 0)
        fr = ",".join(a.get("gen_ai.response.finish_reasons", []))
        extra = f"  [{a['gen_ai.request.model']} · {ti}+{to} tok · finish={fr}]"
    elif "gen_ai.tool.name" in a:
        erro = a.get("error.type")
        extra = f"  [tool={a['gen_ai.tool.name']}" + (f" · ERRO={erro}]" if erro else "]")
    print(f"{ind}└─ {span.name:13s} ({span.span_type:11s}) {span.latency_ms:>5.0f} ms{extra}")
    for filho in span.children:
        imprimir_arvore(filho, nivel + 1)

print("TRACE — raio-x da execução:\n")
for t in tracer.traces:
    imprimir_arvore(t)

## Do trace às métricas: o painel de monitoramento

Além dos spans, as convenções padronizam **métricas agregadas** — `gen_ai.client.operation.duration`
(latência) e `gen_ai.client.token.usage` (consumo). São exatamente os números que um **painel**
consome. Geramos um **lote de traces** (alguns com a ferramenta falhando) e agregamos:
**latência p95** (a cauda lenta que a média esconde), **tokens**, **custo** e **taxa de erro**.

In [ ]:
tracer.reset()
lote = [("Recife", False), ("Salvador", False), ("Natal", False),
        ("Fortaleza", True), ("Maceió", False), ("Recife", False)]  # 1 com timeout
for i, (cidade, falhar) in enumerate(lote):
    responder("Previsão do tempo do meu pedido?", cidade, falhar=falhar, ruido_ms=40 * i)

def percentil(vs, p):
    s = sorted(vs)
    return s[min(len(s) - 1, round(p / 100 * (len(s) - 1)))] if s else 0

def tokens_do_trace(span):
    t = (span.attributes.get("gen_ai.usage.input_tokens", 0)
         + span.attributes.get("gen_ai.usage.output_tokens", 0))
    return t + sum(tokens_do_trace(c) for c in span.children)

def teve_erro(span):
    return ("error.type" in span.attributes) or any(teve_erro(c) for c in span.children)

CUSTO_POR_1K = 0.003
lat = [t.latency_ms for t in tracer.traces]
tok = sum(tokens_do_trace(t) for t in tracer.traces)
n = len(tracer.traces)

print("=== PAINEL OPERACIONAL (agregado a partir dos traces) ===")
print("traces              :", n)
print("latencia_p50_ms     :", percentil(lat, 50))
print("latencia_p95_ms     :", percentil(lat, 95))
print("tokens_total        :", tok)
print("custo_estimado_usd  :", round(tok / 1000 * CUSTO_POR_1K, 4))
print("taxa_erro           :", round(sum(teve_erro(t) for t in tracer.traces) / n, 2))

## Do trace à avaliação: fechando o ciclo

O **mesmo trace** que dá observabilidade alimenta a **validação**. Dele extraímos a
**trajetória** (a sequência de *tool calls*) — exatamente o que um *scorer* agêntico ou um
`@scorer` de código recebe no campo `trace` do `mlflow.genai.evaluate()`.

```python
# PRODUÇÃO — avaliar sobre os traces REAIS de produção:
import mlflow
traces = mlflow.search_traces(experiment_ids=[...])      # puxa os traces gravados
mlflow.genai.evaluate(data=traces, scorers=[meu_juiz_de_trajetoria, usou_ferramenta_certa])
```

In [ ]:
def trajetoria_do_trace(span):
    """Extrai a sequência de ferramentas chamadas (a trajetória) percorrendo os spans TOOL."""
    seq = []
    if span.span_type == SpanType.TOOL:
        seq.append(span.attributes.get("gen_ai.tool.name"))
    for filho in span.children:
        seq += trajetoria_do_trace(filho)
    return seq

# @scorer (code-based) que lê o trace — o elo com o notebook de scorers:
def usou_ferramenta_certa(trace, esperada="consultar_clima"):
    return esperada in trajetoria_do_trace(trace)

for i, t in enumerate(tracer.traces, 1):
    seq = trajetoria_do_trace(t)
    print(f"[trace {i}] trajetoria={seq}  -> usou_ferramenta_certa={usou_ferramenta_certa(t)}")

## Avaliação offline × online

A distinção que organiza toda a prática:

- **Offline** — roda contra *datasets* curados, antes do deploy, na esteira de **CI**. Virtude:
  **repetibilidade** (mesmo conjunto, a cada alteração) para comparar versões e pegar regressões.
- **Online** — roda sobre o **tráfego real** (os traces de produção), em tempo (quase) real.
  Captura o que nenhum dataset previu: o *edge-case* inédito, o pico de custo, o comportamento
  inesperado. Usa monitores *reference-free* + feedback do usuário.

As duas se completam num **ciclo virtuoso**: um trace que **falha em produção (online)** vira um
**caso de dataset**; cria-se um avaliador para ele; valida-se a correção **offline**; e faz-se o
re-deploy. Abaixo, filtramos os traces *online* que **estouram um limiar** — candidatos a virar
caso de teste.

In [ ]:
LIMIAR_P95_MS = 2000

suspeitos = [t for t in tracer.traces if t.latency_ms > LIMIAR_P95_MS or teve_erro(t)]
print(f"{len(suspeitos)} trace(s) suspeito(s) (lat > {LIMIAR_P95_MS} ms ou erro) "
      f"-> viram casos de dataset para avaliação offline:\n")
for t in suspeitos:
    print(f"  - lat={t.latency_ms:.0f} ms | erro={teve_erro(t)} | "
          f"trajetoria={trajetoria_do_trace(t)} | resposta={t.outputs['resposta']!r}")

## (Opcional) O mesmo agente no MLflow de verdade

Até aqui a árvore existia **só na memória do notebook**. Esta seção usa a **API real**
(`@mlflow.trace`, `mlflow.start_span`) no **mesmo** `responder` — ainda sem chamar um LLM, para
não depender de chave. O que muda: os traces são **gravados** no tracking store.

- **Local:** sqlite em `aula-06/.mlflow/` (gitignored). Abra a UI com o comando que a célula
  imprime e vá à aba **Traces** do experimento `aula06-tracing`.
- **Databricks:** o tracking já é o do workspace; os traces aparecem no experimento do notebook.

A latência aqui é a do processo local (quase zero): o valor didático é **ver a árvore na UI**,
não os milissegundos inventados nas seções anteriores. `pip install "mlflow>=3.6"` se ainda não
estiver no ambiente.

In [ ]:
import os, warnings, logging
from pathlib import Path
import mlflow
from mlflow.entities import SpanType as MSpan
from dotenv import load_dotenv

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)
load_dotenv()

def _pasta_aula06() -> Path:
    try:
        d = Path(__file__).resolve().parent
    except NameError:
        d = Path.cwd().resolve()
    for p in [d, *d.parents]:
        if p.name == "aula-06":
            return p
        if (p / "aula-06").is_dir():
            return p / "aula-06"
    return d

if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    mlflow.set_experiment("aula06-tracing")
    ambiente = "Databricks (workspace)"
    uri_ui = None
else:
    _store = _pasta_aula06() / ".mlflow"
    _store.mkdir(parents=True, exist_ok=True)
    _db = (_store / "mlflow.db").resolve()
    _artifacts = (_store / "mlartifacts").resolve()
    _artifacts.mkdir(exist_ok=True)
    mlflow.set_tracking_uri("sqlite:///" + _db.as_posix())
    if mlflow.get_experiment_by_name("aula06-tracing") is None:
        mlflow.create_experiment("aula06-tracing", artifact_location=_artifacts.as_uri())
    mlflow.set_experiment("aula06-tracing")
    ambiente = f"local (sqlite: {_db})"
    uri_ui = f"mlflow ui --backend-store-uri sqlite:///{_db.as_posix()} --port 5000"

@mlflow.trace(name="invoke_agent", span_type=MSpan.AGENT)
def responder_mlflow(pergunta, cidade, falhar=False):
    """O mesmo agente das seções anteriores, agora com spans reais do MLflow."""
    root = mlflow.get_current_active_span()
    root.set_inputs({"pergunta": pergunta, "cidade": cidade})

    with mlflow.start_span(name="chat", span_type=MSpan.CHAT_MODEL) as s1:
        s1.set_attributes({"gen_ai.request.model": "claude-sonnet-4-6",
                           "gen_ai.usage.input_tokens": 180,
                           "gen_ai.usage.output_tokens": 22,
                           "gen_ai.response.finish_reasons": ["tool_calls"]})

    with mlflow.start_span(name="execute_tool", span_type=MSpan.TOOL) as s2:
        s2.set_inputs({"cidade": cidade})
        s2.set_attributes({"gen_ai.tool.name": "consultar_clima"})
        if falhar:
            s2.set_attributes({"error.type": "TimeoutError"})
            out = "Não consegui obter a previsão agora."
            root.set_outputs({"resposta": out})
            return out
        s2.set_outputs(consultar_clima(cidade))

    with mlflow.start_span(name="chat", span_type=MSpan.CHAT_MODEL) as s3:
        s3.set_attributes({"gen_ai.request.model": "claude-sonnet-4-6",
                           "gen_ai.usage.input_tokens": 240,
                           "gen_ai.usage.output_tokens": 34,
                           "gen_ai.response.finish_reasons": ["stop"]})

    resposta = f"Em {cidade}, a previsão é de sol. Bom passeio!"
    root.set_outputs({"resposta": resposta})
    return resposta


def imprimir_trace_mlflow(trace):
    filhos, raizes = {}, []
    for s in trace.data.spans:
        (raizes if not s.parent_id else filhos.setdefault(s.parent_id, [])).append(s)

    def walk(s, nivel=0):
        dur = (s.end_time_ns - s.start_time_ns) / 1e6 if s.end_time_ns else 0
        extra, a = "", (s.attributes or {})
        if "gen_ai.request.model" in a:
            fr = a.get("gen_ai.response.finish_reasons") or []
            extra = (f"  [{a['gen_ai.request.model']} · "
                     f"{a.get('gen_ai.usage.input_tokens', 0)}+"
                     f"{a.get('gen_ai.usage.output_tokens', 0)} tok · "
                     f"finish={','.join(fr) if isinstance(fr, list) else fr}]")
        elif "gen_ai.tool.name" in a:
            erro = a.get("error.type")
            extra = f"  [tool={a['gen_ai.tool.name']}" + (f" · ERRO={erro}]" if erro else "]")
        print(f"{'    '*nivel}└─ {s.name:13s} ({s.span_type:11s}) {dur:>5.0f} ms{extra}")
        for c in filhos.get(s.span_id, []):
            walk(c, nivel + 1)

    print(f"trace_id={trace.info.trace_id}")
    for r in raizes:
        walk(r)


print(responder_mlflow("Como está o tempo na cidade do meu pedido?", "Recife"))
print(responder_mlflow("Previsão do tempo do meu pedido?", "Fortaleza", falhar=True))
mlflow.flush_trace_async_logging()

exp = mlflow.get_experiment_by_name("aula06-tracing")
df = mlflow.search_traces(locations=[exp.experiment_id], max_results=2)
print("\nMLflow", mlflow.__version__, "| tracking:", ambiente)
print("experimento:", exp.name, "| id:", exp.experiment_id, "\n")
for tid in df["trace_id"]:
    imprimir_trace_mlflow(mlflow.get_trace(tid))
    print()
if uri_ui:
    print("Abra a UI e clique em Traces:")
    print(" ", uri_ui)
    print(f"  http://127.0.0.1:5000/#/experiments/{exp.experiment_id}/traces")
else:
    print("No Databricks: Experiments → aula06-tracing → aba Traces.")

### Autolog (opcional²)

Com `ANTHROPIC_API_KEY` no `.env`, uma linha de *autolog* instrumenta a chamada ao Claude: o
span `CHAT_MODEL` nasce sozinho. Compare com a árvore que montamos à mão. Sem a chave, a célula
só avisa e segue — os traces manuais acima já estão na UI.

In [ ]:
if not os.getenv("ANTHROPIC_API_KEY"):
    print("Sem ANTHROPIC_API_KEY — autolog pulado. Recarregue a UI para ver os traces manuais.")
else:
    import anthropic
    mlflow.anthropic.autolog()
    client = anthropic.Anthropic()
    modelo = os.getenv("LLM_MODEL", "claude-sonnet-4-6")
    with mlflow.start_span(name="invoke_agent", span_type=MSpan.AGENT) as root:
        root.set_inputs({"pergunta": "Como está o tempo em Recife?"})
        msg = client.messages.create(
            model=modelo, max_tokens=80,
            messages=[{"role": "user",
                       "content": "Em uma frase, diga o clima típico de Recife em outubro."}],
        )
        texto = msg.content[0].text
        root.set_outputs({"resposta": texto})
    mlflow.flush_trace_async_logging()
    print(texto)
    print()
    imprimir_trace_mlflow(mlflow.get_trace(mlflow.get_last_active_trace_id()))
    print("\nTrace do autolog gravado — recarregue a aba Traces na UI.")

## Conclusão

Você instrumentou um agente com **MLflow Tracing** seguindo as **GenAI semantic conventions do
OpenTelemetry** e fechou o ciclo de observabilidade:

- **Árvore de spans** `invoke_agent` → `chat` / `execute_tool`, cada um com atributos
  **`gen_ai.*`** (`request.model`, `usage.*_tokens`, `response.finish_reasons`, `tool.name`).
- **Trace → métricas:** o mesmo trace agrega num **painel** (latência **p95**, tokens, custo, taxa de erro).
- **Trace → avaliação:** dele extraímos a **trajetória** que alimenta os *scorers* do
  `mlflow.genai.evaluate()` (notebook `exercicio_eval_observabilidade.ipynb`).
- **Offline × online:** um trace que falha em produção vira caso de dataset — o laço entre o que
  acontece e o que se mede.

Adotar um **padrão aberto** (OpenTelemetry) evita o aprisionamento: instrumente uma vez e troque
a ferramenta de visualização sem re-instrumentar. Sobre esse padrão firmou-se um ecossistema —
**MLflow** (Tracing + `mlflow.genai.evaluate`), **LangSmith** e **AgentOps** —, mas o substrato é
sempre o mesmo: **o trace**. Observar, validar e monitorar não são três ferramentas separadas, e
sim **um único ciclo que roda sobre o mesmo dado**.

A seção opcional no final grava essa mesma árvore no **MLflow de verdade** (`@mlflow.trace` +
UI). Com `ANTHROPIC_API_KEY`, o *autolog* (`mlflow.anthropic.autolog()`) ainda mostra os spans
`CHAT_MODEL` nascendo sozinhos — o contraste com o tracer didático.